# ☁️ Model 05 — Weather-Aware AQI Prediction
**AirVintage Production ML Pipeline — Deployment-Ready**

This is AirVintage's **most powerful real-time model** because it combines live pollutant readings with live weather data — both freely available.

| Input | Source | Free? |
|-------|--------|-------|
| `PM2.5`, `PM10` | OpenAQ / CPCB API | ✅ |
| `NO2`, `SO2`, `CO`, `O3` | OpenAQ / CPCB API | ✅ |
| `Temp`, `Humidity` | Open-Meteo / OpenWeatherMap | ✅ |
| `Wind Speed`, `Wind Direction` | Open-Meteo / OpenWeatherMap | ✅ |
| `Pressure`, `Rainfall` | Open-Meteo / OpenWeatherMap | ✅ |
| `Station ID` | CPCB station list | ✅ |
| `Datetime` | System clock | ✅ |

**Dropped:** `NO`, `NOx`, `NH3`, `Benzene`, `Toluene`, `Xylene`

In [ ]:
import sys, os, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna, joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from airvintage_ml import (
    aqi_to_bucket, health_advisory,
    add_temporal_features, add_weather_features,
    encode_categorical, fillna_production, add_entity_stats,
    compute_metrics, print_metrics_table,
    plot_actual_vs_pred, plot_residuals,
    plot_feature_importance, plot_learning_curve_xgb,
    update_model_registry,
)

DATASET='air_cleaned.csv'; MODEL_KEY='air_weather'
MODELS_DIR='models'; REGISTRY=f'{MODELS_DIR}/model_registry.json'; SEED=42
os.makedirs(MODELS_DIR, exist_ok=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)

CORE_POLLUTANTS = ['PM2.5','PM10','NO2','SO2','CO','O3']
WEATHER_INPUTS  = ['Temp_C','Humidity','Wind_Speed','Wind_Direction','Pressure_hPa','Rain_mm']

print(f'XGBoost {xgb.__version__}')
print(f'Pollutant inputs: {CORE_POLLUTANTS}')
print(f'Weather inputs  : {WEATHER_INPUTS}')

In [ ]:
df = pd.read_csv(DATASET, parse_dates=['DateTime'])
# Normalize column names
df = df.rename(columns={'PM2_5':'PM2.5', 'AQI_Target':'AQI'})
print(f'Shape: {df.shape[0]:,} x {df.shape[1]}')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# ── Weather-Focused EDA
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sample = df.sample(min(5000, len(df)), random_state=42)

sc = axes[0,0].scatter(sample['Temp_C'], sample['AQI'], c=sample['AQI'],
                        cmap='RdYlGn_r', alpha=0.4, s=10)
plt.colorbar(sc, ax=axes[0,0], label='AQI')
axes[0,0].set_title('AQI vs Temperature', fontweight='bold')
axes[0,0].set_xlabel('Temp (C)')

sc2 = axes[0,1].scatter(sample['Humidity'], sample['AQI'], c=sample['AQI'],
                          cmap='RdYlGn_r', alpha=0.4, s=10)
plt.colorbar(sc2, ax=axes[0,1])
axes[0,1].set_title('AQI vs Humidity', fontweight='bold')

sc3 = axes[0,2].scatter(sample['Wind_Speed'], sample['AQI'], c=sample['AQI'],
                          cmap='RdYlGn_r', alpha=0.4, s=10)
plt.colorbar(sc3, ax=axes[0,2])
axes[0,2].set_title('AQI vs Wind Speed', fontweight='bold')

df['Is_Rainy'] = (df['Rain_mm'] > 0.1)
for label, grp in df.groupby('Is_Rainy'):
    axes[1,0].hist(grp['AQI'].dropna(), bins=60, alpha=0.65,
                   label='Rainy' if label else 'Dry', edgecolor='white',
                   color='#4facfe' if label else '#f12711')
axes[1,0].legend(); axes[1,0].set_title('AQI: Rainy vs Dry', fontweight='bold')

weather_pol = [c for c in CORE_POLLUTANTS+WEATHER_INPUTS+['AQI'] if c in df.columns]
corr = df[weather_pol].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1,1], linewidths=0.3, annot_kws={'size':7})
axes[1,1].set_title('Correlation Heatmap', fontweight='bold')

all_inputs = CORE_POLLUTANTS + WEATHER_INPUTS
avail = [c for c in all_inputs if c in df.columns]
corr_aqi = df[avail+['AQI']].corr()['AQI'].drop('AQI').abs().sort_values()
colors = ['#4facfe' if c in WEATHER_INPUTS else '#f093fb' for c in corr_aqi.index]
axes[1,2].barh(corr_aqi.index, corr_aqi.values, color=colors)
axes[1,2].set_title('Correlation with AQI\n(blue=weather, pink=pollutant)', fontweight='bold')

fig.suptitle('Air Weather EDA — Production Feature Set', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/eda_05_air_weather.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Engineering — Production Only
df_feat = df.copy()
df_feat = add_temporal_features(df_feat, 'DateTime')

# Pollutant interactions
df_feat['PM_ratio'] = df_feat['PM2.5'] / (df_feat['PM10'] + 1e-6)
df_feat['Total_PM'] = df_feat['PM2.5'] + df_feat['PM10']
df_feat['NO2_SO2']  = df_feat['NO2']   + df_feat['SO2']
df_feat['CO_O3']    = df_feat['CO']    * df_feat['O3']

# Weather-derived features (computed internally, no extra API)
df_feat = add_weather_features(df_feat)

# Station encoding
df_feat['Station_ID'] = df_feat['Station_ID'].astype(str)
df_feat, le_station  = encode_categorical(df_feat, 'Station_ID')
df_feat, stn_stats   = add_entity_stats(df_feat, 'Station_ID_encoded', 'AQI')

DROP = ['DateTime','AQI','Station_ID','Is_Rainy',
        'NO','NOx','NH3','Benzene','Toluene','Xylene']
FEATURE_COLS = [c for c in df_feat.columns if c not in DROP]

df_clean = df_feat.dropna(subset=['AQI']).copy()
df_clean = fillna_production(df_clean)
X, y = df_clean[FEATURE_COLS], df_clean['AQI']

weather_in_features = [c for c in FEATURE_COLS if c in WEATHER_INPUTS + ['Wind_NS','Wind_EW','Heat_Index','Pressure_norm','Is_Rainy','Is_Calm']]
print(f'Production features ({len(FEATURE_COLS)}) total')
print(f'Weather features   : {weather_in_features}')
print(f'Samples            : {X.shape[0]:,}')

In [ ]:
X_train,X_tmp,y_train,y_tmp = train_test_split(X,y,test_size=0.30,random_state=SEED)
X_val,X_test,y_val,y_test   = train_test_split(X_tmp,y_tmp,test_size=0.50,random_state=SEED)
print(f'Train:{X_train.shape[0]:,} | Val:{X_val.shape[0]:,} | Test:{X_test.shape[0]:,}')

In [ ]:
def objective(trial):
    p = {
        'n_estimators'    : trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate'   : trial.suggest_float('lr', 0.01, 0.2, log=True),
        'max_depth'       : trial.suggest_int('max_depth', 4, 10),
        'min_child_weight': trial.suggest_int('mcw', 1, 10),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('csb', 0.5, 1.0),
        'reg_alpha'       : trial.suggest_float('alpha', 1e-4, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('lam', 1e-4, 10.0, log=True),
        'tree_method':'hist','random_state':SEED,'verbosity':0
    }
    m = xgb.XGBRegressor(**p, early_stopping_rounds=30, eval_metric='rmse')
    m.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=False)
    return float(np.sqrt(np.mean((y_val-m.predict(X_val))**2)))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=40, show_progress_bar=True)
best_params = study.best_params
print(f'Best RMSE: {study.best_value:.4f}')

In [ ]:
model = xgb.XGBRegressor(**best_params,tree_method='hist',random_state=SEED,
                           early_stopping_rounds=50,eval_metric='rmse',verbosity=0)
model.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)

cv_p = dict(best_params, n_estimators=model.best_iteration or 500)
cv_m = xgb.XGBRegressor(**cv_p, tree_method='hist', random_state=SEED, verbosity=0)
kr2 = cross_val_score(cv_m,X,y,cv=KFold(5,shuffle=True,random_state=SEED),scoring='r2',n_jobs=-1)
print(f'5-Fold CV R2: {kr2.mean():.4f} +/- {kr2.std():.4f}')

In [ ]:
train_m=compute_metrics(y_train,model.predict(X_train))
val_m  =compute_metrics(y_val,  model.predict(X_val))
test_m =compute_metrics(y_test, model.predict(X_test))
print_metrics_table({'Train':train_m,'Val':val_m,'Test':test_m})

plot_learning_curve_xgb(model.evals_result(),model.best_iteration,'XGBoost — Weather-Aware',f'{MODELS_DIR}/lc_05_air_weather.png')
plot_actual_vs_pred(y_test,model.predict(X_test),'Actual vs Pred — Weather-Aware',f'{MODELS_DIR}/avp_05_air_weather.png')
plot_residuals(y_test,model.predict(X_test),'Residuals — Weather-Aware',f'{MODELS_DIR}/res_05_air_weather.png')
plot_feature_importance(FEATURE_COLS,model.feature_importances_,'Feature Importance — Weather-Aware',savepath=f'{MODELS_DIR}/fi_05_air_weather.png')

# Weather features importance breakdown
fi_df = pd.DataFrame({'Feature':FEATURE_COLS,'Importance':model.feature_importances_})
fi_w  = fi_df[fi_df['Feature'].isin(WEATHER_INPUTS+['Wind_NS','Wind_EW','Heat_Index','Pressure_norm','Is_Rainy','Is_Calm'])]
fi_w  = fi_w.sort_values('Importance')
plt.figure(figsize=(9,4))
plt.barh(fi_w['Feature'], fi_w['Importance'], color=plt.cm.Blues(np.linspace(0.4,0.9,len(fi_w))))
plt.title('Weather Feature Importance (subset)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/fi_05_weather_only.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
model.save_model(f'{MODELS_DIR}/05_air_weather_xgb.json')
joblib.dump(le_station, f'{MODELS_DIR}/05_air_weather_le_station.pkl')
stn_stats.to_csv(f'{MODELS_DIR}/05_air_weather_stn_stats.csv', index=False)
with open(f'{MODELS_DIR}/05_air_weather_features.json','w') as f: json.dump(FEATURE_COLS,f,indent=2)
with open(f'{MODELS_DIR}/05_air_weather_input_schema.json','w') as f:
    json.dump({'pollutant_inputs':CORE_POLLUTANTS,
               'weather_inputs':WEATHER_INPUTS,
               'other_inputs':['station_id','datetime'],
               'pollutant_source':'OpenAQ / CPCB free API',
               'weather_source':'Open-Meteo / OpenWeatherMap free tier'},f,indent=2)

update_model_registry(REGISTRY,MODEL_KEY,'XGBoost+Weather+Optuna',DATASET,test_m,
    model_paths={'model':f'{MODELS_DIR}/05_air_weather_xgb.json','encoder':f'{MODELS_DIR}/05_air_weather_le_station.pkl',
                 'stn_stats':f'{MODELS_DIR}/05_air_weather_stn_stats.csv','features':f'{MODELS_DIR}/05_air_weather_features.json'},
    feature_count=len(FEATURE_COLS),notes='Best real-time model. Inputs: PM2.5,PM10,NO2,SO2,CO,O3 + 6 weather params (all free)')
print('Model 05 saved.')

In [ ]:
def predict_weather_aware(
    station_id: str, datetime_str: str,
    # Pollutants — from OpenAQ/CPCB free API
    pm25: float, pm10: float, no2: float, so2: float, co: float, o3: float,
    # Weather — from Open-Meteo / OpenWeatherMap free tier
    temp_c: float, humidity: float, wind_speed: float,
    wind_dir: float, pressure_hpa: float, rain_mm: float
) -> dict:
    """
    Predict AQI with full weather + pollutant context.

    Pollutants  : pm25, pm10, no2, so2, co, o3           (OpenAQ / CPCB free)
    Weather     : temp_c, humidity, wind_speed, wind_dir, (Open-Meteo / OWM free)
                  pressure_hpa, rain_mm
    """
    row = pd.DataFrame([{
        'Station_ID': station_id, 'DateTime': datetime_str,
        'PM2.5': pm25, 'PM10': pm10, 'NO2': no2, 'SO2': so2, 'CO': co, 'O3': o3,
        'Temp_C': temp_c, 'Humidity': humidity, 'Wind_Speed': wind_speed,
        'Wind_Direction': wind_dir, 'Pressure_hPa': pressure_hpa, 'Rain_mm': rain_mm
    }])
    row['DateTime'] = pd.to_datetime(row['DateTime'])
    row = add_temporal_features(row, 'DateTime')
    row['PM_ratio']=pm25/(pm10+1e-6); row['Total_PM']=pm25+pm10
    row['NO2_SO2']=no2+so2; row['CO_O3']=co*o3
    row = add_weather_features(row)

    row['Station_ID'] = str(station_id)
    s_enc = int(le_station.transform([str(station_id)])[0]) if str(station_id) in le_station.classes_ else -1
    row['Station_ID_encoded'] = s_enc
    sr = stn_stats[stn_stats['Station_ID_encoded']==s_enc]
    for col in [c for c in stn_stats.columns if 'AQI' in c]:
        row[col] = float(sr[col].values[0]) if len(sr)>0 else 150.0

    X_in = row[FEATURE_COLS].fillna(0)
    aqi  = float(model.predict(X_in)[0])
    b    = aqi_to_bucket(aqi)
    return {
        'model':'air_weather','station_id':station_id,'datetime':datetime_str,
        'predicted_aqi':round(aqi,2),'aqi_category':b,'health_advisory':health_advisory(b),
        'weather_context':{'temp_c':temp_c,'humidity':humidity,'wind_speed':wind_speed,'rain_mm':rain_mm}
    }

demo = predict_weather_aware(
    station_id='1', datetime_str='2025-01-15 14:00:00',
    pm25=90, pm10=140, no2=50, so2=12, co=1.5, o3=25,
    temp_c=28, humidity=65, wind_speed=3.5, wind_dir=270, pressure_hpa=1010, rain_mm=0.0
)
print('Demo (Weather-Aware):'); [print(f'  {k}: {v}') for k,v in demo.items()]